## Sistemas de Recomendación 1/2

Vamos a ver sistemas de recomendacion en dos ejemplos:

1. Ejemplos básicos basandos en popularidad y Content Based
2. basado en filtro colaborativo y basando en modelos (SVD y KNN) aplicando el paquete Surprise

### Estructura del notebook

| Sección | Técnica | Tipo |
|---|---|---|
| 3 | Popularidad ponderada (Bayesian weighted rating) | No personalizado |
| 4 | Content-Based (similitud del coseno sobre tags) | Personalizado |

> **Dataset**: GoodBooks-10k — 10.000 libros, ~6M ratings de 53.424 usuarios. 
Ratings de 1 a 5. Incluye metadatos de libros y tags de usuarios.

> ⚠️ **Nota de compatibilidad**: La librería `surprise` (usada en el notebook 2/2) 
requiere `numpy < 2.0`. Si tienes numpy 2.x, ejecuta: `pip install 'numpy<2' scikit-surprise`

In [1]:
### Ojo surprise SVD funciona con una version de numpy<2 

#### Importamos las librerías necesarias

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import os

In [2]:
os.getcwd()

'c:\\Users\\tomas\\ML\\Master Data Science and AI\\08 Sistemas de recomendacion'

#### 0. Importamos los datos

**Data set obtenido de kaggle**

https://www.kaggle.com/datasets/zygmunt/goodbooks-10k

There have been good datasets for movies (Netflix, Movielens) and music (Million Songs) recommendation, but not for books. That is, until now.

This dataset contains ratings for ten thousand popular books. As to the source, let's say that these ratings were found on the internet. Generally, there are 100 reviews for each book, although some have less - fewer - ratings. Ratings go from one to five.

Both book IDs and user IDs are contiguous. For books, they are 1-10000, for users, 1-53424. All users have made at least two ratings. Median number of ratings per user is 8.

There are also books marked to read by the users, book metadata (author, year, etc.) and tags.

In [3]:
ratings_data = pd.read_csv('./data/ratings.csv')
books_metadata = pd.read_csv('./data/books.csv')
ratings_data.head(10)

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
5,1,2077,4
6,1,2487,4
7,1,2900,5
8,1,3662,4
9,1,3922,5


El dataset `ratings.csv` contiene tres columnas: `book_id`, `user_id` y `rating`. 
Es la **matriz de interacción usuario-ítem** en formato largo (una fila por valoración).

`books.csv` contiene los metadatos: título, autor, año, ISBN, imagen de portada, etc.

#### 1. Exploracion de los datos

Antes de modelar conviene entender la densidad de la matriz usuario-ítem: 
cuántos usuarios hay, cuántos libros, y cómo se distribuyen las valoraciones. 
Una matriz muy dispersa (*sparse*) puede limitar la calidad de los modelos colaborativos.

In [4]:
books_metadata.head(10)

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...
5,6,11870085,11870085,16827462,226,525478817,9.780525e+12,John Green,2012.0,The Fault in Our Stars,...,2346404,2478609,140739,47994,92723,327550,698471,1311871,https://images.gr-assets.com/books/1360206420m...,https://images.gr-assets.com/books/1360206420s...
6,7,5907,5907,1540236,969,618260307,9.780618e+12,J.R.R. Tolkien,1937.0,The Hobbit or There and Back Again,...,2071616,2196809,37653,46023,76784,288649,665635,1119718,https://images.gr-assets.com/books/1372847500m...,https://images.gr-assets.com/books/1372847500s...
7,8,5107,5107,3036731,360,316769177,9.780317e+12,J.D. Salinger,1951.0,The Catcher in the Rye,...,2044241,2120637,44920,109383,185520,455042,661516,709176,https://images.gr-assets.com/books/1398034300m...,https://images.gr-assets.com/books/1398034300s...
8,9,960,960,3338963,311,1416524797,9.781417e+12,Dan Brown,2000.0,Angels & Demons,...,2001311,2078754,25112,77841,145740,458429,716569,680175,https://images.gr-assets.com/books/1303390735m...,https://images.gr-assets.com/books/1303390735s...
9,10,1885,1885,3060926,3455,679783261,9.780680e+12,Jane Austen,1813.0,Pride and Prejudice,...,2035490,2191465,49152,54700,86485,284852,609755,1155673,https://images.gr-assets.com/books/1320399351m...,https://images.gr-assets.com/books/1320399351s...


In [5]:
ratings_data.shape

(981756, 3)

In [6]:
ratings_data["book_id"].value_counts().head().to_frame()

,count
book_id,
1,100
5198,100
5215,100
5214,100
9803,100


In [7]:
rating_counts = pd.DataFrame(ratings_data["book_id"].value_counts())
rating_counts

,count
book_id,
1,100
5198,100
5215,100
5214,100
9803,100
...,...
9315,36
1935,34
9486,24


In [8]:
n_users = ratings_data.user_id.unique().shape[0]
n_items = ratings_data.book_id.unique().shape[0]
print(n_users, n_items)

53424 10000


#### 2. Preparación de los datos
junto los ratings con los metadata de los libros para tener en una misma tabla los nombres de los libros

Hacemos un **merge** entre los ratings y los metadatos para poder mostrar títulos y autores. 
Usamos `book_id` de ratings con `id` de books (¡ojo: columnas con nombres distintos!).

Tras el merge, la columna `book_id` del resultado se llama `book_id_x` porque hay colisión de nombres.

In [10]:
df = ratings_data.merge(books_metadata, left_on="book_id", right_on="id", how="inner") 
# ojo , id del metadato es el book_id de los ratings
df

,book_id_x,user_id,rating,id,book_id_y,best_book_id,work_id,books_count,isbn,isbn13,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,314,5,1,2767052,2767052,2792775,272,439023483,9.780439e+12,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,1,439,3,1,2767052,2767052,2792775,272,439023483,9.780439e+12,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
2,1,588,5,1,2767052,2767052,2792775,272,439023483,9.780439e+12,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
3,1,1169,4,1,2767052,2767052,2792775,272,439023483,9.780439e+12,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
4,1,1185,4,1,2767052,2767052,2792775,272,439023483,9.780439e+12,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
981751,10000,48386,5,10000,8914,8914,11817,31,375700455,9.780376e+12,...,9162,9700,364,117,345,2031,4138,3069,https://images.gr-assets.com/books/1403194704m...,https://images.gr-assets.com/books/1403194704s...
981752,10000,49007,4,10000,8914,8914,11817,31,375700455,9.780376e+12,...,9162,9700,364,117,345,2031,4138,3069,https://images.gr-assets.com/books/1403194704m...,https://images.gr-assets.com/books/1403194704s...
981753,10000,49383,5,10000,8914,8914,11817,31,375700455,9.780376e+12,...,9162,9700,364,117,345,2031,4138,3069,https://images.gr-assets.com/books/1403194704m...,https://images.gr-assets.com/books/1403194704s...
981754,10000,50124,5,10000,8914,8914,11817,31,375700455,9.780376e+12,...,9162,9700,364,117,345,2031,4138,3069,https://images.gr-assets.com/books/1403194704m...,https://images.gr-assets.com/books/1403194704s...


Comprobamos que el numero de usuarios y libros es correcto. Y vemos que no se han duplicado registros. 

In [11]:
n_users = df.user_id.unique().shape[0]
n_items = df.book_id_x.unique().shape[0]
print(n_users, n_items)

53424 10000


In [12]:
conteo_usuarios = df["user_id"].value_counts()
conteo_usuarios

user_id
30944    200
12874    200
12381    199
52036    199
28158    199
        ... 
10351      2
16592      2
24343      2
41314      2
27590      2
Name: count, Length: 53424, dtype: int64

In [13]:
conteo_books = df["book_id_x"].value_counts()
conteo_books

book_id_x
1       100
5198    100
5215    100
5214    100
9803    100
       ... 
9315     36
1935     34
9486     24
9345     11
7803      8
Name: count, Length: 10000, dtype: int64

#### 3. Modelo basado en popularidad

Usamos la **fórmula de Bayesian Weighted Rating** (la misma que usa IMDB para su Top 250):

$$WR = \frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C$$

Donde:
- $v$ = número de votos del libro
- $m$ = mínimo de votos para entrar en el ranking (percentil 90)
- $R$ = media de ratings del libro
- $C$ = media global de ratings de todos los libros --> C= df['rating'].mean

**¿Por qué no ordenar directamente por media?** Un libro con 3 valoraciones de 5★ 
aparecería por encima de uno con 500 valoraciones de 4.8★. La fórmula pondera la confianza.
Lo que hace esta formula es cuanto mas votos tiene mas pesa su propio rating. 
para libros con pocos votos, su rating pesa la mita y la media global de ratings la otra mitad.

In [14]:
C= df['rating'].mean()
C

3.8565335989797873

In [16]:
# Agrupar por USER_ID y calcular conteo + media
df_books = df.groupby("book_id_x")["rating"].agg(["count", "mean"]).reset_index()
df_books

,book_id_x,count,mean
0,1,100,4.240000
1,2,100,4.210000
2,3,100,3.090000
3,4,100,4.460000
4,5,100,3.890000
...,...,...,...
9995,9996,98,3.969388
9996,9997,89,4.426966
9997,9998,95,4.326316
9998,9999,99,3.727273


In [17]:
m= df_books['count'].quantile(0.9)
m

100.0

In [18]:
#WR = (v/(v+m))*R+(m/(v+m))*C
df_books['new_rating']=(df_books['count']/(df_books['count']+m))*df_books['mean']+((m/(df_books['count']+m))*C)
df_books

,book_id_x,count,mean,new_rating
0,1,100,4.240000,4.048267
1,2,100,4.210000,4.033267
2,3,100,3.090000,3.473267
3,4,100,4.460000,4.158267
4,5,100,3.890000,3.873267
...,...,...,...,...
9995,9996,98,3.969388,3.912391
9996,9997,89,4.426966,4.125150
9997,9998,95,4.326316,4.085402
9998,9999,99,3.727273,3.792228


Ahora ordenamos por new_rating

In [19]:
df_books.sort_values(by="new_rating", ascending=False).head(10)  # ✅ usar new_rating

,book_id_x,count,mean,new_rating
6919,6920,100,4.780000,4.318267
5206,5207,100,4.780000,4.318267
9565,9566,99,4.777778,4.314841
6360,6361,100,4.770000,4.313267
3274,3275,100,4.770000,4.313267
7946,7947,89,4.820225,4.310335
5579,5580,100,4.750000,4.303267
6589,6590,100,4.750000,4.303267
4482,4483,100,4.750000,4.303267
8945,8946,93,4.774194,4.298722


Entonces, basandose en popularity, recomendaré a todos los usuarios los 5 libros con mayor valoracion y que no haya leido aun.

La función filtra los libros que el usuario **ya ha leído** y devuelve el top-N 
de los mejor valorados (según `new_rating`) que aún no conoce.

Es el modelo más sencillo: no personaliza por gustos, solo excluye lo ya leído.

In [21]:
def get_recommendacion_pop(user_id, top):
    libros_leidos = df[(df["user_id"] == user_id)]["book_id_x"].to_frame()
    df_filtrado = df_books[~df_books["book_id_x"].isin(libros_leidos["book_id_x"])]
    df_top = df_filtrado.sort_values(by="new_rating", ascending=False).head(top)  # ✅ usar new_rating
    df_top2= df_top.merge(books_metadata, left_on="book_id_x", right_on="id", how="inner")
    return df_top2[["title", "id", "new_rating", "authors"]]

Top 10 recomendaciones para el usuario 1000

In [22]:
get_recommendacion_pop(1000, 10)

,title,id,new_rating,authors
0,The Days Are Just Packed: A Calvin and Hobbes ...,5207,4.318267,Bill Watterson
1,The Indispensable Calvin and Hobbes,6920,4.318267,Bill Watterson
2,Attack of the Deranged Mutant Killer Monster S...,9566,4.314841,Bill Watterson
3,There's Treasure Everywhere: A Calvin and Hobb...,6361,4.313267,Bill Watterson
4,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",3275,4.313267,"J.K. Rowling, Mary GrandPré"
5,ESV Study Bible,7947,4.310335,"Anonymous, Lane T. Dennis, Wayne A. Grudem"
6,The Calvin and Hobbes Lazy Sunday Book,5580,4.303267,Bill Watterson
7,The Authoritative Calvin and Hobbes: A Calvin ...,6590,4.303267,Bill Watterson
8,It's a Magical World: A Calvin and Hobbes Coll...,4483,4.303267,Bill Watterson
9,The Divan,8946,4.298722,Hafez


Top 10 recomendaciones para el usuario 1001

In [23]:
get_recommendacion_pop(1001, 10)

,title,id,new_rating,authors
0,The Days Are Just Packed: A Calvin and Hobbes ...,5207,4.318267,Bill Watterson
1,The Indispensable Calvin and Hobbes,6920,4.318267,Bill Watterson
2,Attack of the Deranged Mutant Killer Monster S...,9566,4.314841,Bill Watterson
3,There's Treasure Everywhere: A Calvin and Hobb...,6361,4.313267,Bill Watterson
4,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",3275,4.313267,"J.K. Rowling, Mary GrandPré"
5,ESV Study Bible,7947,4.310335,"Anonymous, Lane T. Dennis, Wayne A. Grudem"
6,The Calvin and Hobbes Lazy Sunday Book,5580,4.303267,Bill Watterson
7,The Authoritative Calvin and Hobbes: A Calvin ...,6590,4.303267,Bill Watterson
8,It's a Magical World: A Calvin and Hobbes Coll...,4483,4.303267,Bill Watterson
9,The Divan,8946,4.298722,Hafez


#### 4. Modelo Content Based

### ¿Cómo funciona?

1. Cada libro tiene asociadas **tags** (etiquetas puestas por los usuarios: género, temática, etc.)
2. Construimos una **matriz libro × tag** (binaria: 1 si el libro tiene esa tag, 0 si no)
3. Calculamos la **similitud del coseno** entre todos los pares de libros
4. Para recomendar: dado un libro que le gustó al usuario, devolvemos los más similares

> ⚠️ La pivot table y el cálculo de similitud del coseno son **operaciones costosas** 
(varios minutos). La matriz libro×tag puede tener miles de columnas.

Se basa en buscar similitud entre diferentes libros, en base a palabras clave, genero, etc.  
De tal manera que a los usuarios que les ha gustado el elemento A les genera recomendaciones parecidas a A de acuerdo a palabras clave, genero, etc  
Aqui utilizo una tabla del data set que es book_tags donde cada palabra clave ya ha sido etiquetada con un valor numérico
luego haré simulitud del coseno en base a estas palabras clave y finalmente obtenenemos la recomendacion en base a los libros mas similares.

In [24]:
print(books_metadata.columns) 

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='object')


In [25]:
book_tags = pd.read_csv('./data/book_tags.csv')
book_tags

,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716
...,...,...,...
999907,33288638,21303,7
999908,33288638,17271,7
999909,33288638,1126,7
999910,33288638,11478,7


In [26]:
tags = pd.read_csv('./data/tags.csv')
tags

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-
...,...,...
34247,34247,Ｃhildrens
34248,34248,Ｆａｖｏｒｉｔｅｓ
34249,34249,Ｍａｎｇａ
34250,34250,ＳＥＲＩＥＳ


In [27]:
book_tags2= book_tags.merge(books_metadata, left_on="goodreads_book_id", right_on="book_id", how="inner")[["id", "tag_id"]]
book_tags2

,id,tag_id
0,27,30574
1,27,11305
2,27,11557
3,27,8717
4,27,33114
...,...,...
999907,8892,21303
999908,8892,17271
999909,8892,1126
999910,8892,11478


Para hacerlo un poco mas visual me traido el valor de las tags, pero podría hacerse perfectamente con los tag_id. la pivot table tarda unos minutos

In [28]:
book_tags3= book_tags2.merge(tags, left_on="tag_id", right_on="tag_id", how="inner")[["id", "tag_name"]]
book_tags3

,id,tag_name
0,27,to-read
1,27,fantasy
2,27,favorites
3,27,currently-reading
4,27,young-adult
...,...,...
999907,8892,neighbors
999908,8892,kindleunlimited
999909,8892,5-star-reads
999910,8892,fave-author


Tarda bastante construir la pivot table directamente y puede dar error de memoria  

books_tags_df = book_tags3.pivot_table(index=["id"], columns=["tag_name"], aggfunc=lambda x: 1, fill_value=0)

Por ello filtramos por las tags mas frecuentes


Que hace crs_matrix:  
La idea clave es que en tu matriz la mayoría de valores son 0 — un libro solo tiene unas pocas decenas de tags de las 10.000 posibles.  
Almacenar todos esos ceros es un desperdicio enorme.  
csr_matrix (Compressed Sparse Row) solo guarda los valores distintos de cero y su posición. El resto se asume 0.

In [30]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── 1. Filtrar tags poco frecuentes ──────────────────────────────────
# Nos quedamos solo con tags que aparecen en al menos 20 libros
# Reduce las columnas de ~10.000 a ~1.000-2.000
min_apariciones = 20
tags_frecuentes = book_tags3.groupby("tag_name")["id"].count()
tags_frecuentes = tags_frecuentes[tags_frecuentes >= min_apariciones].index
book_tags_filtrado = book_tags3[book_tags3["tag_name"].isin(tags_frecuentes)]

print(f"Tags originales:  {book_tags3['tag_name'].nunique()}")
print(f"Tags tras filtro: {len(tags_frecuentes)}")

# ── 2. Pivot con int8 en vez de int64 (x8 menos memoria) ─────────────
books_tags_df = book_tags_filtrado.pivot_table(
    index=["id"],
    columns=["tag_name"],
    aggfunc=lambda x: 1,
    fill_value=0
).astype(np.int8)   # 0/1 → int8 es suficiente

print(f"Matriz: {books_tags_df.shape}  "
      f"Memoria: {books_tags_df.memory_usage().sum() / 1e6:.1f} MB")

# ── 3. Convertir a sparse antes de calcular similitud ─────────────────
# sklearn cosine_similarity acepta sparse matrices directamente
sparse_matrix = csr_matrix(books_tags_df.values)
sparse_matrix

Tags originales:  34252
Tags tras filtro: 3359
Matriz: (10000, 3359)  Memoria: 33.7 MB


<Compressed Sparse Row sparse matrix of dtype 'int8'
	with 909498 stored elements and shape (10000, 3359)>

In [31]:
sparse_matrix

<Compressed Sparse Row sparse matrix of dtype 'int8'
	with 909498 stored elements and shape (10000, 3359)>

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

El calculo de la similarity lleva algunos minutos

La **similitud del coseno** mide el ángulo entre dos vectores de tags. 
Valor 1 = libros idénticos en tags, valor 0 = sin tags en común.

La matriz resultante es **simétrica de NxN** (N = número de libros únicos con tags). 
⏱️ Este cálculo puede tardar varios minutos.

In [32]:
cosine_sim = cosine_similarity(sparse_matrix, sparse_matrix)

print(f"cosine_sim shape: {cosine_sim.shape}")

cosine_sim shape: (10000, 10000)


In [33]:
cosine_sim

array([[1.        , 0.53334035, 0.55211329, ..., 0.3471283 , 0.23288574,
        0.17825592],
       [0.53334035, 1.        , 0.49746943, ..., 0.36837621, 0.18956823,
        0.18916706],
       [0.55211329, 0.49746943, 1.        , ..., 0.24553428, 0.13905533,
        0.19213069],
       ...,
       [0.3471283 , 0.36837621, 0.24553428, ..., 1.        , 0.20198059,
        0.32011384],
       [0.23288574, 0.18956823, 0.13905533, ..., 0.20198059, 1.        ,
        0.26845243],
       [0.17825592, 0.18916706, 0.19213069, ..., 0.32011384, 0.26845243,
        1.        ]])

#### 5. Funciones
Son unas funciones para obtener informacion del libro. Las usaremos también en los modelos SVD  (Single Value Decomposition)

In [34]:
import difflib
import random

def get_book_id(book_title, metadata):
    
    """
    Gets the book ID for a book title based on the closest match in the metadata dataframe.
    """
    
    existing_titles = list(metadata['title'].values)
    closest_titles = difflib.get_close_matches(book_title, existing_titles)
    book_id = metadata[metadata['title'] == closest_titles[0]]['id'].values[0]
    return book_id

def get_book_info(book_id, metadata):
    
    """
    Returns some basic information about a book given the book id and the metadata dataframe.
    """
    
    book_info = metadata[metadata['id'] == book_id][['id', 'isbn', 
                                                    'authors', 'title', 'original_title']]
    return book_info.to_dict(orient='records')


> ⚠️ **Error en `recomendar_libro`**: La función usa `sim_matrix[idx]` donde `idx` es el `book_id` real 
(ej. 2508), pero `sim_matrix` está indexada **posicionalmente** (fila 0, 1, 2…). 
Si los IDs no son consecutivos desde 0, accede a la fila incorrecta.

> Además, `similares2['id']` contiene índices posicionales, no `book_id`s reales, 
por lo que el merge final devuelve resultados incorrectos.

> **Versión corregida** abajo:

#### 5. Funciones auxiliares

- `get_book_id`: dado un título (con posibles erratas), busca el más parecido usando `difflib` y devuelve su ID
- `get_book_info`: devuelve metadatos básicos de un libro dado su ID
- `recomendar_libro`: función principal de recomendación content-based

In [37]:
def recomendar_libro(titulo, df, sim_matrix):
    book_id = get_book_id(titulo, df)
    idx     = books_tags_df.index.get_loc(book_id)

    similitudes = list(enumerate(sim_matrix[idx]))
    similares   = sorted(similitudes, key=lambda x: x[1], reverse=True)[1:4]

    book_ids_recomendados = [books_tags_df.index[i[0]] for i in similares]
    scores                = [round(i[1], 4) for i in similares]

    recomendados = df[df['id'].isin(book_ids_recomendados)][['title', 'id', 'authors']].copy()
    
    # Añadimos el score respetando el orden del ranking
    score_map = dict(zip(book_ids_recomendados, scores))
    recomendados['cosine_similarity'] = recomendados['id'].map(score_map)
    recomendados = recomendados.sort_values('cosine_similarity', ascending=False)

    return recomendados[['title', 'authors', 'cosine_similarity']]


In [38]:
# **Ejemplo**: Recomendar libros similares a "Libro A"
libros_recomendados = recomendar_libro("ESV Study Bible", books_metadata, cosine_sim)
print("Libros recomendados:", libros_recomendados['title'])
libros_recomendados

Libros recomendados: 4777    The Holy Bible: English Standard Version
5967                      The Jesus I Never Knew
2235       Holy Bible: New International Version
Name: title, dtype: object


,title,authors,cosine_similarity
4777,The Holy Bible: English Standard Version,Anonymous,0.6428
5967,The Jesus I Never Knew,Philip Yancey,0.6144
2235,Holy Bible: New International Version,Anonymous,0.5909


---
### Resumen y limitaciones

| Modelo | Ventaja | Limitación |
|---|---|---|
| **Popularidad** | Simple, funciona con usuarios nuevos | No personalizado: todos reciben lo mismo |
| **Content-Based** | Personaliza por similitud de contenido | Cold-start para libros nuevos sin tags; no descubre géneros nuevos |

El notebook **2/2** abordará el **filtro colaborativo** (SVD, KNN) con la librería `surprise`, 
que aprende de los patrones de valoración de todos los usuarios.